# 5.3 Medallion Architecture (Bronze / Silver / Gold)

Reference notebook walking through the medallion layering pattern using SAP-style data, with explicit mapping to the SAP BW layered model so the transition feels familiar.

| Medallion | SAP BW analogue | Purpose |
|-----------|----------------|---------|
| Bronze | PSA (Persistent Staging Area) | Raw ingest, schema-on-read, append-only |
| Silver | DSO / ADSO (cleansed, deduped) | Conformed, business-key-stable, query-ready |
| Gold | InfoCube / Composite Provider / Query | Aggregated marts, KPIs, dimensional models |

Below: a single SAP sales flow processed across the three layers using `pandas` + `sqlite3`. PySpark/Delta equivalents are annotated.

In [ ]:
import pandas as pd
import sqlite3
from io import StringIO

raw = StringIO('''VBELN,POSNR,MATNR,KUNNR,WERKS,NETWR,KWMENG,VDATU,STATUS
0000010001,10,MAT-00001,CUST-0001,1000,1500.00,100,2025-09-15,OPEN
0000010001,20,MAT-00002,CUST-0001,1000, ,40,2025-09-15,OPEN
0000010002,10,MAT-00001,CUST-0002,2000,2200.00,150,2025-09-18,SHIPPED
0000010002,10,MAT-00001,CUST-0002,2000,2200.00,150,2025-09-18,SHIPPED
0000010003,10,MAT-00003,CUST-0003,1000,800.00,300,2025-09-22,CANCELLED
''')
bronze = pd.read_csv(raw)
bronze

## Bronze: raw ingest

- Append-only, source-shape preserved
- Add ingestion metadata (load timestamp, source system, file name)
- No business logic, no filters

In [ ]:
bronze['_ingest_ts'] = pd.Timestamp('2025-09-23 02:00:00')
bronze['_source'] = 'ECC.VBAP'
bronze
# PySpark/Delta:
# (df.withColumn('_ingest_ts', F.current_timestamp())
#    .withColumn('_source', F.lit('ECC.VBAP'))
#    .write.format('delta').mode('append').saveAsTable('bronze.sap_vbap'))

## Silver: cleansed and conformed

- Drop exact duplicates
- Cast types, normalize nulls
- Apply data quality expectations (e.g., NETWR not null, STATUS in valid set)
- Stable business keys (here: `VBELN` + `POSNR`)

In [ ]:
silver = (bronze
          .drop_duplicates(subset=['VBELN', 'POSNR'])
          .assign(NETWR=lambda x: pd.to_numeric(x['NETWR'], errors='coerce'))
          .assign(VDATU=lambda x: pd.to_datetime(x['VDATU']))
          .query("STATUS in ['OPEN', 'SHIPPED', 'CANCELLED']")
          .dropna(subset=['NETWR']))
silver
# PySpark/DLT:
# @dlt.table
# @dlt.expect_or_drop('netwr_not_null', 'NETWR IS NOT NULL')
# @dlt.expect_or_drop('valid_status', "STATUS IN ('OPEN','SHIPPED','CANCELLED')")
# def silver_sales_orders():
#     return (dlt.read('bronze_sap_vbap')
#             .dropDuplicates(['VBELN','POSNR'])
#             .withColumn('NETWR', F.col('NETWR').cast('decimal(15,2)'))
#             .withColumn('VDATU', F.to_date('VDATU')))

## Gold: aggregated marts

- Business-facing, dimensionally modelled
- Pre-aggregated to support KPIs / dashboards
- One row per (month, plant) here — typical BW InfoCube grain

In [ ]:
gold = (silver
        .assign(CALMONTH=silver['VDATU'].dt.to_period('M').astype(str))
        .groupby(['CALMONTH', 'WERKS'])
        .agg(open_value=('NETWR', lambda s: s[silver.loc[s.index, 'STATUS'] == 'OPEN'].sum()),
             shipped_value=('NETWR', lambda s: s[silver.loc[s.index, 'STATUS'] == 'SHIPPED'].sum()),
             order_count=('VBELN', 'nunique'))
        .reset_index())
gold
# PySpark equivalent uses conditional sums:
# silver.groupBy('CALMONTH','WERKS').agg(
#     F.sum(F.when(F.col('STATUS')=='OPEN',    F.col('NETWR'))).alias('open_value'),
#     F.sum(F.when(F.col('STATUS')=='SHIPPED', F.col('NETWR'))).alias('shipped_value'),
#     F.countDistinct('VBELN').alias('order_count'))

## Mapping back to SAP BW

If you've designed BW flows before, the mental model maps cleanly:

| BW concept | Medallion equivalent |
|------------|----------------------|
| DataSource / PSA | Bronze table |
| Transformation / Routine | Silver build logic |
| ADSO (write-optimized) | Bronze (append) |
| ADSO (standard, with activation) | Silver (merge / dedupe) |
| InfoCube / aggregate | Gold table |
| Composite Provider | Gold view across multiple Gold tables |
| BEx Query | Databricks SQL view / dashboard |

Key differences to internalize:
- Schema-on-read (Bronze) vs locked DataSource structure
- Delta Lake handles the activation/merge that BW activation queues used to manage
- Gold is just a table — the OLAP / aggregate layer is a query engine concern, not modeled as cubes